# Epi Info AI Space-Time Cluster Detection inference lab — V0.3

Independently reconstruct the deterministic candidate-window scoring stage for `EPIAI CLUSTER SPACE_TIME`. This notebook reads the raw synthetic CSV and manifest and does not call the TypeScript implementation under test.

## Boundary

This lab verifies case-only retrospective space-time permutation margins, circular spatial candidates, temporal windows, expected counts, high-cluster likelihood ranking, and seeded maximum-statistic Monte Carlo inference. It independently reproduces the browser candidate without importing its TypeScript implementation.

In [ ]:
UINT32_MASK = 0xffffffff
def imul32(left, right):
    return ((left & UINT32_MASK) * (right & UINT32_MASK)) & UINT32_MASK

def mulberry32(seed):
    state = seed & UINT32_MASK
    def random():
        nonlocal state
        state = (state + 0x6d2b79f5) & UINT32_MASK
        value = state
        value = imul32(value ^ (value >> 15), value | 1)
        value = (value ^ ((value + imul32(value ^ (value >> 7), value | 61)) & UINT32_MASK)) & UINT32_MASK
        return ((value ^ (value >> 14)) & UINT32_MASK) / 4294967296
    return random

def shuffled(values, random):
    result = list(values)
    for index in range(len(result) - 1, 0, -1):
        replacement = math.floor(random() * (index + 1))
        result[index], result[replacement] = result[replacement], result[index]
    return result

def monte_carlo_maxima(source, replications):
    study_start = date.fromisoformat(options['START']); total = len(source)
    prepared = sorted(({**row, 'day': (date.fromisoformat(row[options['DATE']]) - study_start).days, 'point': (float(row[options['LATITUDE']]), float(row[options['LONGITUDE']]))} for row in source), key=lambda row: row[options['ID']])
    by_point = defaultdict(list)
    for row in prepared: by_point[row['point']].append(row)
    max_spatial_cases = total * float(options['MAXCASEFRACTION']); spatial = {}
    for center in sorted(by_point):
        ordered = sorted((haversine(center, point), point) for point in by_point if haversine(center, point) <= float(options['MAXDISTANCEKM']) + 1e-12)
        members, cases = [], 0
        for radius, point in ordered:
            if cases + len(by_point[point]) > max_spatial_cases + 1e-12: break
            members.append(point); cases += len(by_point[point]); key = tuple(sorted(members))
            if key not in spatial or radius < spatial[key]['radius']: spatial[key] = {'radius': radius, 'members': key, 'cases': cases}
    windows = [spatial[key] for key in sorted(spatial)]
    time_bins = (date.fromisoformat(options['END']) - study_start).days + 1
    max_length = max(1, min(int(options['MAXTIMEUNITS']), math.floor(time_bins * float(options['MAXTIMEFRACTION']))))
    temporal = [(first, last, sum(first <= row['day'] <= last for row in prepared)) for first in range(time_bins) for last in range(first, min(time_bins, first + max_length))]
    memberships = [[row['point'] in window['members'] for row in prepared] for window in windows]
    original_bins = [row['day'] for row in prepared]; random = mulberry32(int(options['SEED'])); maxima = []
    for _ in range(replications):
        permuted = shuffled(original_bins, random); maximum = 0.0
        for first, last, time_cases in temporal:
            for window, membership in zip(windows, memberships):
                observed = sum(inside and first <= bin_number <= last for inside, bin_number in zip(membership, permuted))
                expected = window['cases'] * time_cases / total
                maximum = max(maximum, llr(observed, expected, total))
        maxima.append(maximum)
    return maxima, len(windows) * len(temporal) * replications

In [ ]:
import csv, hashlib, io, json, math, sys
from collections import defaultdict
from datetime import date, timedelta
import scipy
from pyodide.http import pyfetch

manifest_response = await pyfetch('../../validation-fixtures/space-time-cluster-synthetic-v0.1.json')
manifest_response.raise_for_status(); manifest = await manifest_response.json()
csv_response = await pyfetch('../../validation-fixtures/space-time-cluster-synthetic-v0.1.csv')
csv_response.raise_for_status(); csv_bytes = await csv_response.bytes()
assert hashlib.sha256(csv_bytes).hexdigest() == manifest['dataset']['sha256']
records = list(csv.DictReader(io.StringIO(csv_bytes.decode('utf-8-sig'))))
assert len(records) == manifest['dataset']['recordCount'] == 30
options = {token.split('=', 1)[0]: token.split('=', 1)[1] for token in manifest['command'].split()[3:]}
{'python': sys.version, 'scipy': scipy.__version__, 'records': len(records), 'state': manifest['state']}

In [ ]:
EARTH_RADIUS_KM = 6371.0088
def haversine(a, b):
    lat1, lon1 = map(math.radians, a); lat2, lon2 = map(math.radians, b)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    chord = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(min(1, math.sqrt(chord)))

def llr(observed, expected, total):
    if observed <= expected or expected <= 0 or expected >= total: return 0.0
    outside_observed, outside_expected = total - observed, total - expected
    outside = outside_observed * math.log(outside_observed / outside_expected) if outside_observed else 0.0
    return max(0.0, observed * math.log(observed / expected) + outside)

def score(source):
    study_start = date.fromisoformat(options['START']); total = len(source)
    prepared = [{**row, 'day': (date.fromisoformat(row[options['DATE']]) - study_start).days, 'point': (float(row[options['LATITUDE']]), float(row[options['LONGITUDE']]))} for row in source]
    by_point = defaultdict(list)
    for row in prepared: by_point[row['point']].append(row)
    max_spatial_cases = total * float(options['MAXCASEFRACTION']); spatial = {}
    for center in sorted(by_point):
        ordered = sorted(((haversine(center, point), point) for point in by_point if haversine(center, point) <= float(options['MAXDISTANCEKM']) + 1e-12))
        members, cases = [], 0
        for radius, point in ordered:
            if cases + len(by_point[point]) > max_spatial_cases + 1e-12: break
            members.append(point); cases += len(by_point[point]); key = tuple(sorted(members))
            if key not in spatial or radius < spatial[key]['radius']: spatial[key] = {'center': center, 'radius': radius, 'members': key, 'cases': cases}
    time_bins = (date.fromisoformat(options['END']) - study_start).days + 1
    max_length = max(1, min(int(options['MAXTIMEUNITS']), math.floor(time_bins * float(options['MAXTIMEFRACTION']))))
    temporal = [sum(first <= row['day'] <= last for row in prepared) for first in range(time_bins) for last in range(first, min(time_bins, first + max_length))]
    scored, temporal_index = [], 0
    for first in range(time_bins):
        for last in range(first, min(time_bins, first + max_length)):
            time_cases = temporal[temporal_index]; temporal_index += 1
            for window in spatial.values():
                inside = [row for row in prepared if row['point'] in window['members'] and first <= row['day'] <= last]
                expected = window['cases'] * time_cases / total; score_value = llr(len(inside), expected, total)
                if score_value > 0: scored.append({'observed': len(inside), 'expected': expected, 'llr': score_value, 'start': str(study_start + timedelta(days=first)), 'end': str(study_start + timedelta(days=last)), 'members': window['members'], 'caseIds': sorted(row[options['ID']] for row in inside)})
    return sorted(scored, key=lambda item: (-item['llr'], -item['observed'], item['start'], item['members']))

independent = score(records); independent[0]

In [ ]:
top = independent[0]; planted = manifest['plantedCluster']
assert top['caseIds'] == planted['caseIds']
assert top['observed'] == 12
assert math.isclose(top['expected'], 5.6, rel_tol=0, abs_tol=1e-12)
assert top['start'] == planted['start'] and top['end'] == planted['end']
assert len(top['members']) == 2
{'status': 'PASS', 'observed': top['observed'], 'expected': top['expected'], 'observedExpectedRatio': top['observed'] / top['expected'], 'logLikelihoodRatio': top['llr'], 'pValue': None}

In [ ]:
reversed_result = score(list(reversed(records)))
renamed_irrelevant = [{**row, 'syndrome': 'RENAMED'} for row in records]
renamed_result = score(renamed_irrelevant)
for result in (reversed_result, renamed_result):
    assert result[0]['caseIds'] == top['caseIds']
    assert math.isclose(result[0]['expected'], top['expected'], rel_tol=0, abs_tol=1e-12)
    assert math.isclose(result[0]['llr'], top['llr'], rel_tol=0, abs_tol=1e-12)
{'row-order invariance': 'PASS', 'irrelevant-field invariance': 'PASS'}

In [ ]:
expected_inference = manifest['expectedInference']; replications = int(options['REPLICATIONS'])
null_maxima, work = monte_carlo_maxima(records, replications)
exceedances = sum(value + 1e-12 >= top['llr'] for value in null_maxima)
p_value = (1 + exceedances) / (1 + replications)
assert expected_inference['randomGenerator'] == 'mulberry32-v1'
assert work == expected_inference['work'] == 1398600
assert exceedances == expected_inference['topCluster']['monteCarloExceedances'] == 12
assert math.isclose(p_value, expected_inference['topCluster']['pValue'], rel_tol=0, abs_tol=1e-15)
assert null_maxima[:20] == monte_carlo_maxima(list(reversed(records)), 20)[0]
{'status': 'PASS', 'randomGenerator': 'mulberry32-v1', 'replications': replications, 'exceedances': exceedances, 'pValue': p_value, 'maximum-statistic adjustment': True}

## Result and next gate

A clean run independently ranks the planted two-location, five-day window first, reproduces its expected count and likelihood score, and reproduces the seeded maximum-statistic result of 12 exceedances in 999 replications (`p=0.013`). External differential evidence, Worker cancellation/progress, `EPIAI CLUSTER RENDER`, and epidemiologist review remain required before execution is enabled.